# 간단 뉴스 크롤링 튜토리얼 - '권영국' 후보 기사 수집

`selenium`을 활용해 특정 후보(권영국)에 대한 네이버 뉴스 기사를 크롤링하는 튜토리얼입니다.
2025년 5월 28일을 최신 날짜로 하여 최대 100건의 기사를 수집하고, CSV 파일로 저장합니다.

In [24]:
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time

from utils import *

import os
import json
from datetime import datetime

# 기본 설정
name = '권영국'
press_code = {
    '경향신문': '1032',
    '동아일보': '1020',
    '오마이뉴스': '1047',
    '조선일보': '1023',
    '중앙일보': '1025',
    '한겨레신문': '1028'
}

# 날짜 설정
start_date = datetime(2025, 5,27)
end_date = datetime(2025, 5, 28)

In [25]:
# 1. 링크를 수집하여 메타 데이터로 저장하는 함수

def get_meta_data(name, press_code, start_date, end_date):
    """
    셀레니움으로 스크롤 끝까지 내려서 해당하는 날짜의 모든 기사 div 로딩 후,
    각 기사의 url을 저장하여 json으로 메타데이터 저장
    """
    start_date_str = start_date.strftime('%Y.%m.%d')
    end_date_str = end_date.strftime('%Y.%m.%d')

    chrome_options = Options()
    # chrome_options.add_argument("--headless")  # 필요시 주석 해제
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=chrome_options)

    result = {}
    result[name] = {}

    curr_date = start_date
    while curr_date <= end_date:
        ds = curr_date.strftime('%Y.%m.%d')
        target_date = curr_date.strftime('%Y%m%d')
        for press_name, code in press_code.items():
            if press_name not in result[name]:
                result[name][press_name] = {'네이버링크': [], '언론사링크': []}

            url = (
                f"https://search.naver.com/search.naver?"
                f"ssc=tab.news.all&query={name}&sm=tab_opt&sort=0&photo=0&field=0"
                f"&pd=3&ds={ds}&de={ds}"
                f"&docid=&related=0&mynews=1"
                f"&office_type=1&office_section_code=4"
                f"&news_office_checked={code}"
                f"&nso=so%3Ar%2Cp%3Afrom{target_date}to{target_date}"
                f"&is_sug_officeid=0&office_category=0&service_area="
            )

            print(f"\n=== [{name} - {press_name} - {ds}] ===")
            driver.get(url)
            time.sleep(2)

            prev_len = 0
            while True:
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(1.5)
                elements = driver.find_elements(By.CLASS_NAME, "group_news")
                curr_len = len(elements)
                if curr_len == prev_len:
                    break
                prev_len = curr_len

            links = driver.find_elements(By.XPATH, '//div[contains(@class, "group_news")]//a[@href and @target="_blank"]')
            for link in links:
                href = link.get_attribute("href")
                if not href or not href.startswith("http"):
                    continue
                if "n.news.naver.com" in href:
                    if href not in result[name][press_name]['네이버링크']:
                        result[name][press_name]['네이버링크'].append(href)
                else:
                    if href not in result[name][press_name]['언론사링크']:
                        result[name][press_name]['언론사링크'].append(href)

        curr_date += timedelta(days=1)

    driver.quit()
    save_json(result, f'meta_data_{start_date_str}_{end_date_str}.json')


In [30]:
# 2.위에서 저장한 URL에 방문하여 본문 파싱하는 함수

from itertools import count

def fetch_and_parse(name, start_date, end_date,
                    raw_dir="raw", batch_size=None):
    """
    meta 파일을 읽어 URL 방문 → 기사 본문 수집 → RAW JSON 저장
    (※ 정렬·슬라이싱·order 재부여는 하지 않음)
    """
    start_date_str = start_date.strftime("%Y.%m.%d")
    end_date_str   = end_date.strftime("%Y.%m.%d")
    meta = load_json(f"meta_data_{start_date_str}_{end_date_str}.json")

    data_raw = []
    ord_gen = count(1)

    for press, link_dict in meta[name].items():
        urls = link_dict.get("네이버링크") or link_dict.get("언론사링크", [])
        for url in urls:
            ord_val = next(ord_gen)
            print(f"=== {name} | {press} | RAW order {ord_val} ===")

            art = parse_crawled(url)                 # 사용자 정의 함수
            ymd = art["date"][:10].replace(".", "")  # YYYYMMDD

            data_raw.append({
                "id"     : f"{name}_{press}_{ymd}_{ord_val}",
                "order"  : ord_val,                  # 수집순
                "title"  : art["title"],
                "press"  : press,
                "date"   : art["date"],
                "content": art["content"],
                "source" : art["source"]
            })

    raw_path = os.path.join(
        raw_dir,
        f"news_raw_{name}_{start_date_str}_{end_date_str}.json"
    )
    save_json(data_raw, raw_path)
    print(f"✅ 원본 기사 {len(data_raw)}건 저장 → {raw_path}")

    return data_raw

In [32]:
# 3. 위에서 수집한 기사 데이터를 날짜 최신순 정렬하여 100건만 남기는 함수

def preprocess_and_save_one(data_list, candidate,
                            output_dir="output", batch_size=100):
    """
    최신순 정렬 → 상위 batch_size 선택 → order 1부터 재부여 → clean JSON 저장
    """
    os.makedirs(output_dir, exist_ok=True)

    data_sorted = sorted(
        data_list, key=lambda x: x["date"], reverse=True
    )[:batch_size]

    for idx, art in enumerate(data_sorted, 1):
        art["order"] = idx

    clean_path = os.path.join(output_dir, f"{candidate}.json")
    save_json(data_sorted, clean_path)
    print(f"✅ {candidate} clean 데이터 {len(data_sorted)}건 저장 → {clean_path}")

In [33]:
# 4. 전체 파이프라인 - main() -> 1,2,3 함수를 차례로 호출

def main(name, press_code, start_date, end_date, batch_size=100):
    print("-" * 50)
    print(f"{start_date.strftime('%Y.%m.%d')} ~ {end_date.strftime('%Y.%m.%d')}")
    print(f"대상 후보: {name}")
    print(f"언론사: {list(press_code.keys())}")
    print("-" * 50)

    print("[STEP 1] 메타데이터 수집")
    get_meta_data(name, press_code, start_date, end_date)

    print("\n[STEP 2] 기사 원본 수집 (RAW)")
    raw_list = fetch_and_parse(name, start_date, end_date)

    print("\n[STEP 3] 최신 100건 정제 (CLEAN)")
    preprocess_and_save_one(raw_list, candidate=name, batch_size=batch_size)

In [34]:
# 5. 실제 실행

if __name__ == '__main__':
    main(name, press_code, start_date, end_date)

--------------------------------------------------
2025.05.27 ~ 2025.05.28
대상 후보: 권영국
언론사: ['경향신문', '동아일보', '오마이뉴스', '조선일보', '중앙일보', '한겨레신문']
--------------------------------------------------
[STEP 1] 메타데이터 수집

=== [권영국 - 경향신문 - 2025.05.27] ===

=== [권영국 - 동아일보 - 2025.05.27] ===

=== [권영국 - 오마이뉴스 - 2025.05.27] ===

=== [권영국 - 조선일보 - 2025.05.27] ===

=== [권영국 - 중앙일보 - 2025.05.27] ===

=== [권영국 - 한겨레신문 - 2025.05.27] ===

=== [권영국 - 경향신문 - 2025.05.28] ===

=== [권영국 - 동아일보 - 2025.05.28] ===

=== [권영국 - 오마이뉴스 - 2025.05.28] ===

=== [권영국 - 조선일보 - 2025.05.28] ===

=== [권영국 - 중앙일보 - 2025.05.28] ===

=== [권영국 - 한겨레신문 - 2025.05.28] ===

[STEP 2] 기사 원본 수집 (RAW)
=== 권영국 | 경향신문 | RAW order 1 ===
=== 권영국 | 경향신문 | RAW order 2 ===
=== 권영국 | 경향신문 | RAW order 3 ===
=== 권영국 | 경향신문 | RAW order 4 ===
=== 권영국 | 경향신문 | RAW order 5 ===
=== 권영국 | 경향신문 | RAW order 6 ===
=== 권영국 | 경향신문 | RAW order 7 ===
=== 권영국 | 경향신문 | RAW order 8 ===
=== 권영국 | 경향신문 | RAW order 9 ===
=== 권영국 | 경향신문 | RAW order 10 ===
=== 권영국 |